In [ ]:
!pip install -q fastapi uvicorn pyngrok pillow transformers accelerate
!pip install -q 'git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2]'
!pip install -q 'av>=12.0.0' --upgrade

DEPRECATION: git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 138.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 79.2 MB/s eta 0:00:00


In [ ]:
# Patch pyav_utils.py so av.option.Option annotation is not evaluated at import time
import glob
for path in glob.glob('/usr/local/lib/python3.*/dist-packages/lerobot/datasets/pyav_utils.py'):
    src = open(path).read()
    if 'from __future__ import annotations' not in src:
        open(path, 'w').write('from __future__ import annotations' + src)
        print(f'Patched: {path}')
    else:
        print(f'Already patched: {path}')

Patched: /usr/local/lib/python3.13/dist-packages/lerobot/datasets/pyav_utils.py


In [ ]:
import os
import threading
import time
import numpy as np
import torch
from google.colab import userdata
from pyngrok import ngrok
from huggingface_hub import login
from transformers import AutoProcessor

# ── Secrets ───────────────────────────────────────────────────────────────────
os.environ['HF_TOKEN']    = userdata.get('HF_TOKEN')
os.environ['NGROK_TOKEN'] = userdata.get('NGROK_TOKEN')

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF login OK')

# ── Load fine-tuned policy ────────────────────────────────────────────────────
from lerobot.policies.molmoact2 import MolmoAct2Policy

REPO_ID    = 'subhodipsaha/molmoact2_stack_three_cubes'
CHUNK_SIZE = 30

print(f'Loading {REPO_ID}...')
t0 = time.time()
policy = MolmoAct2Policy.from_pretrained(REPO_ID)
policy = policy.to('cuda').eval().to(dtype=torch.bfloat16)
print(f'Loaded in {time.time()-t0:.1f}s  |  GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ── Load Molmo processor (lives in the base model repo only) ─────────────────
print('Loading Molmo processor from base model...')
proc = AutoProcessor.from_pretrained('allenai/MolmoAct2-SO100_101', trust_remote_code=True)
print(f'Processor: {type(proc).__name__}')

# ── Inject stack-cubes QUANTILE normalization as a new norm_tag ───────────────
# Stats from policy_preprocessor_step_2_molmoact2_masked_normalizer.safetensors
# (subhodipsaha/molmoact2_stack_three_cubes)
_robot_stats = policy.model._get_robot_stats()
_NormCls     = type(next(iter(_robot_stats.state_normalizers.values())))

def _make_q01_q99_normalizer(q01, q99):
    n = object.__new__(_NormCls)
    n.mode      = 'q01_q99'
    n.mean      = None
    n.std       = None
    n.min_val   = np.array(q01, dtype=np.float32)
    n.max_val   = np.array(q99, dtype=np.float32)
    n.q_low     = np.array(q01, dtype=np.float32)
    n.q_high    = np.array(q99, dtype=np.float32)
    n.mask      = np.ones(len(q01), dtype=bool)
    n.zero_mask = np.zeros(len(q01), dtype=bool)
    return n

_STATE_Q01  = [-4.690, -98.715, 16.555, 21.013, -87.609, 12.128]
_STATE_Q99  = [31.575,  26.274, 96.517, 77.663, -84.848, 47.751]
_ACTION_Q01 = [-4.873, -100.483, 14.087, 20.197, -87.753,  1.136]
_ACTION_Q99 = [31.962,   25.674, 96.955, 78.323, -84.663, 48.121]

NORM_TAG = 'so101_stack_cubes'
_robot_stats.metadata_by_tag[NORM_TAG] = {'action_dim': 6}
_robot_stats.state_normalizers[NORM_TAG]  = _make_q01_q99_normalizer(_STATE_Q01, _STATE_Q99)
_robot_stats.action_normalizers[NORM_TAG] = _make_q01_q99_normalizer(_ACTION_Q01, _ACTION_Q99)

# Patch get_action_dim for custom tag
_orig_get_action_dim = _robot_stats.__class__.get_action_dim
def _patched_get_action_dim(self, norm_tag):
    if str(norm_tag) == NORM_TAG:
        return 6
    return _orig_get_action_dim(self, norm_tag)
_robot_stats.__class__.get_action_dim = _patched_get_action_dim

print(f'Injected norm_tag={NORM_TAG!r}  state=[{_STATE_Q01}, {_STATE_Q99}]')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK
Loading subhodipsaha/molmoact2_stack_three_cubes...


config.json:   0%|          | 0.00/3.47k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

A new version of the following files was downloaded from https://huggingface.co/allenai/MolmoAct2-SO100_101:
- video_processing_molmoact2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


image_processing_molmoact2.py:   0%|          | 0.00/22.4k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/allenai/MolmoAct2-SO100_101:
- image_processing_molmoact2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/allenai/MolmoAct2-SO100_101:
- processing_molmoact2.py
- video_processing_molmoact2.py
- image_processing_molmoact2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`MolmoAct2Processor` defines `image_processor_class = 'AutoImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.
`MolmoAct2Processor` defines `video_processor_class = 'AutoVideoProcessor'`, which is deprecated. Register the correct mapping in `AutoVideoProcessor` instead.


configuration_molmoact2.py:   0%|          | 0.00/20.7k [00:00<?, ?B/s]

Processor: MolmoAct2Processor
Injected norm_tag='so101_stack_cubes'  state=[[-4.69, -98.715, 16.555, 21.013, -87.609, 12.128], [31.575, 26.274, 96.517, 77.663, -84.848, 47.751]]


In [ ]:
# Debug: test predict_action with injected norm_tag
import traceback as _tb
import numpy as np
from PIL import Image as _Img

_phone_pil = _Img.new('RGB', (480, 640))
_wrist_pil = _Img.new('RGB', (640, 480))
# Use HOME_POS as a realistic state test
_state = np.array([-3.9, -98.7, 96.5, 77.4, -86.4, 41.7], dtype=np.float32)

print(f'Testing predict_action with norm_tag={NORM_TAG!r} ...')
try:
    _out = policy.model.predict_action(
        processor=proc,
        images=[_phone_pil, _wrist_pil],
        task='Stack all three cubes into a tower',
        state=_state,
        norm_tag=NORM_TAG,
        inference_action_mode='continuous',
        n_action_steps=CHUNK_SIZE,
        enable_cuda_graph=False,
    )
    _acts = _out.actions
    if _acts.ndim == 3:
        _acts = _acts[0]
    print(f'OK — shape: {_acts.shape}')
    print(f'First: {_acts[0].cpu().float().tolist()}')
    print(f'Last:  {_acts[-1].cpu().float().tolist()}')
except Exception:
    _tb.print_exc()

Testing predict_action with norm_tag='so101_stack_cubes' ...
OK — shape: torch.Size([30, 6])
First: [-3.9375, -100.5, 97.0, 78.5, -86.5, 42.0]
Last:  [-3.71875, -99.5, 96.5, 46.0, -86.5, 41.5]


In [ ]:
import base64, io, time, os
import numpy as np
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import uvicorn

PORT = 8000
LOG  = '/content/molmoact2_stack_server.log'
_logf = open(LOG, 'a', buffering=1)

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    _logf.write(line + '')

app = FastAPI()

class PredictRequest(BaseModel):
    images_b64: List[str]
    state: List[float]
    task: str

class PredictResponse(BaseModel):
    actions: List[List[float]]
    inference_ms: float

_server_ready = False

@app.get('/health')
def health():
    return {'status': 'ok', 'gpu_mem_gb': round(torch.cuda.memory_allocated()/1e9, 2),
            'warmed_up': _server_ready}

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest):
    global _server_ready
    t0 = time.time()

    phone_pil = Image.open(io.BytesIO(base64.b64decode(req.images_b64[0]))).convert('RGB')
    wrist_pil = Image.open(io.BytesIO(base64.b64decode(req.images_b64[1]))).convert('RGB')
    state_np  = np.array(req.state, dtype=np.float32)

    log(f"predict | state={[round(v,1) for v in req.state]}")

    # enable_cuda_graph=False: CUDA graphs freeze input buffers at capture time,
    # causing the model to silently replay warmup inputs for every subsequent call.
    output = policy.model.predict_action(
        processor=proc,
        images=[phone_pil, wrist_pil],
        task=req.task,
        state=state_np,
        norm_tag=NORM_TAG,
        inference_action_mode='continuous',
        n_action_steps=CHUNK_SIZE,
        enable_cuda_graph=False,
    )

    _server_ready = True
    actions = output.actions.cpu().float().numpy()
    if actions.ndim == 3:
        actions = actions[0]

    ms = (time.time() - t0) * 1000
    log(f"  → shape={actions.shape}  {ms:.0f}ms  first={np.round(actions[0], 2).tolist()}")
    return PredictResponse(actions=actions.tolist(), inference_ms=round(ms, 1))

# ── Restartable server lifecycle ──────────────────────────────────────────────
_uvicorn_server = None
_server_thread  = None

def start_server():
    global _uvicorn_server, _server_thread

    if _uvicorn_server is not None:
        log('Stopping previous server...')
        _uvicorn_server.should_exit = True
        _server_thread.join(timeout=5)
        time.sleep(0.5)

    try:
        ngrok.kill()
    except Exception:
        pass

    config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
    _uvicorn_server = uvicorn.Server(config)
    _server_thread  = threading.Thread(target=_uvicorn_server.run, daemon=True)
    _server_thread.start()
    time.sleep(2)

    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
    tunnel = ngrok.connect(PORT, 'http')
    log(f'Server ready | {tunnel.public_url}')
    print('' + '='*60)
    print(f'  NGROK URL: {tunnel.public_url}')
    print(f'  → paste into config.py as MOLMOACT2_STACK_SERVER')
    print('='*60 + '')

start_server()

[11:10:07] Server ready | https://21c7-35-240-194-21.ngrok-free.app
  NGROK URL: https://21c7-35-240-194-21.ngrok-free.app
  → paste into config.py as MOLMOACT2_STACK_SERVER


In [ ]:
# Tail live server logs
!tail -f /content/molmoact2_stack_server.log

[11:18:20] predict | state=[-3.9, -98.7, 96.5, 77.4, -86.4, 41.7]
[11:18:21]   → shape=(30, 6)  1102ms  first=[-3.940000057220459, -100.5, 97.0, 78.5, -86.5, 42.0]
[11:19:44] predict | state=[-3.6, -98.2, 98.2, 77.1, -86.8, 41.4]
[11:19:45]   → shape=(30, 6)  1067ms  first=[-3.5799999237060547, -99.5, 97.0, 78.0, -87.0, 41.75]
[11:19:47] predict | state=[-3.6, -99.5, 97.0, 78.0, -87.0, 41.5]
[11:19:48]   → shape=(30, 6)  1064ms  first=[-3.359999895095825, -100.5, 97.0, 78.5, -87.0, 42.0]
[11:19:49] predict | state=[-3.4, -100.0, 97.0, 78.5, -87.0, 42.0]
[11:19:50]   → shape=(30, 6)  1040ms  first=[-3.299999952316284, -100.5, 97.0, 78.5, -87.0, 42.5]
[11:19:51] predict | state=[-3.3, -100.5, 97.0, 78.5, -87.0, 42.2]
[11:19:52]   → shape=(30, 6)  1050ms  first=[-3.2200000286102295, -100.5, 97.0, 78.5, -87.0, 42.75]
[11:19:53] predict | state=[-3.3, -100.0, 97.0, 78.5, -87.0, 43.0]
[11:19:54]   → shape=(30, 6)  1051ms  first=[-3.2200000286102295, -100.5, 97.0, 78.5, -87.0, 43.5]
[11:19:55